In [1]:
from pyspark.sql import SparkSession   # create spark session
from pyspark.sql.functions import col  # column operations

spark = SparkSession.builder.appName("ItemCF").getOrCreate()

# load ratings dataset
ratings = spark.read.csv(
    "/home/rajesh/CSL7100/Assignment3/ml-latest-small/ratings.csv",
    header=True,
    inferSchema=True
)

ratings.show(5)


26/03/15 11:50:13 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/15 11:50:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 11:50:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows



In [2]:

# create user-item matrix
user_item = ratings.select("userId", "movieId", "rating")

user_item.show(5)

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   4.0|
|     1|      3|   4.0|
|     1|      6|   4.0|
|     1|     47|   5.0|
|     1|     50|   5.0|
+------+-------+------+
only showing top 5 rows



In [3]:
print((ratings.count(), len(ratings.columns)))

(100836, 4)


In [7]:
from pyspark.sql.functions import col   # column operations

# create two copies of ratings
ratings1 = ratings.alias("r1")
ratings2 = ratings.alias("r2")

# join where same user rated both movies
movie_pairs = ratings1.join(
    ratings2,
    (col("r1.userId") == col("r2.userId")) &
    (col("r1.movieId") < col("r2.movieId"))   # avoid duplicate pairs
)

movie_pairs = movie_pairs.select(
    #col("r1.userId").alias("userId"),
    col("r1.movieId").alias("movie1"),
    col("r2.movieId").alias("movie2"),
    col("r1.rating").alias("rating1"),
    col("r2.rating").alias("rating2")
)

movie_pairs.show(5)


+------+------+-------+-------+
|movie1|movie2|rating1|rating2|
+------+------+-------+-------+
|     1|  5060|    4.0|    5.0|
|     1|  4006|    4.0|    4.0|
|     1|  3809|    4.0|    4.0|
|     1|  3793|    4.0|    5.0|
|     1|  3744|    4.0|    4.0|
+------+------+-------+-------+
only showing top 5 rows



## 1. Item to item using Cosine similarity.

In [5]:
print("no of movies: ", (movie_pairs.count(), " no of Users: ", len(movie_pairs.columns)))

no of movies:  (30396650, ' no of Users: ', 4)


In [8]:
from pyspark.sql.functions import sum, sqrt

pair_stats = movie_pairs.groupBy("movie1","movie2").agg(

    sum(col("rating1") * col("rating2")).alias("dot_product"),

    sqrt(sum(col("rating1") * col("rating1"))).alias("norm1"),

    sqrt(sum(col("rating2") * col("rating2"))).alias("norm2")

)

pair_stats.show(5)

26/03/15 11:58:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:58:56 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:58:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:58:58 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:03 WARN RowBasedKeyValueBatch: Calling spill() on

+------+------+-----------+------------------+------------------+
|movie1|movie2|dot_product|             norm1|             norm2|
+------+------+-----------+------------------+------------------+
|     1|  1348|     137.25|13.047988350699889|11.022703842524301|
|     1|  1208|      901.5|29.111853256019273| 32.11697370550345|
|     3|   367|      324.5|18.614510468986285|18.808242873804026|
|     6|  2028|     1040.0|  31.3966558728792|33.926390907374746|
|     6|  1219|     433.75|20.838665984174707|21.523243250030884|
+------+------+-----------+------------------+------------------+
only showing top 5 rows



In [11]:
print((pair_stats.count(), len(pair_stats.columns)))

[Stage 35:>                                                       (0 + 12) / 13]

(13157672, 5)


In [9]:
similarity = pair_stats.withColumn(
    "similarity",
    col("dot_product") / (col("norm1") * col("norm2"))
)

similarity.select("movie1","movie2","similarity").show(10)

26/03/15 11:59:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:57 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 11:59:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 12:00:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 12:00:00 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 12:00:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 12:00:02 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/15 12:00:03 WARN RowBasedKeyValueBatch: Calling spill() on

+------+------+------------------+
|movie1|movie2|        similarity|
+------+------+------------------+
|     1|  1348|0.9542906174099952|
|     1|  1208|0.9641869773819718|
|     3|   367|0.9268615311585415|
|     6|  2028|0.9763651840118103|
|     6|  1219|0.9670787940282444|
|   101|  2268|0.9774079958981905|
|   101|  2033|0.9980732612097091|
|   110|  1298|0.9709657290405586|
|   110|   736|0.9579565116565327|
|   157|  2414|0.9091372900969896|
+------+------+------------------+
only showing top 10 rows



In [10]:
c

[Stage 28:===================================================>    (12 + 1) / 13]

(13157672, 6)


In [12]:
target_user = 10


In [14]:

from pyspark.sql.functions import col   # column operations

user_movies = ratings.filter(
    col("userId") == target_user
).select("movieId","rating")

user_movies.show(5)

+-------+------+
|movieId|rating|
+-------+------+
|    296|   1.0|
|    356|   3.5|
|    588|   4.0|
|    597|   3.5|
|    912|   4.0|
+-------+------+
only showing top 5 rows



In [ ]:
#Join User Movies with Item Similarity

In [ ]:
similar_items = similarity.join(
    user_movies,
    similarity.movie1 == user_movies.movieId
)
similar_items.show(5)

In [15]:
print((similar_items.count(), len(similar_items.columns)))

NameError: name 'similar_items' is not defined

In [ ]:

similar_items = similar_items.select(
    col("movie2").alias("movieId"),
    col("similarity"),
    col("rating")
)

similar_items.show(10)

In [ ]:
from pyspark.sql.functions import sum
#Compute Weighted Score
weighted_scores = similar_items.withColumn(
    "weighted_rating",
    col("similarity") * col("rating")
)

In [ ]:
#Aggregate to Predict Ratings
predicted_ratings = weighted_scores.groupBy("movieId").agg(

    (sum("weighted_rating") / sum("similarity")).alias("predicted_rating")

)

predicted_ratings.show(10)

In [ ]:
# Remove Movies Already Rated
predicted_ratings = predicted_ratings.join(
    user_movies,
    on="movieId",
    how="left_anti"   # removes already watched movies
)

## 3. Generate top-N movie recommendations for a user

In [ ]:
from pyspark.sql.functions import col   # column operations

# number of recommendations
N = 10

# sort predicted ratings
top_recommendations = predicted_ratings.orderBy(
    col("predicted_rating").desc()
).limit(N)

top_recommendations.show()

In [16]:
#Evaluate the System (RMSE)
train_ratings, test_ratings = ratings.randomSplit([0.8,0.2], seed=42)
predictions = predicted_ratings.join(
    test_ratings.select("userId","movieId","rating"),
    on="movieId"
)

In [ ]:
from pyspark.sql.functions import pow, avg, sqrt

rmse_df = predictions.withColumn(
    "squared_error",
    pow(col("rating") - col("predicted_rating"),2)
)

rmse = rmse_df.select(
    sqrt(avg("squared_error")).alias("RMSE")
)

rmse.show()

In [ ]:
#Precision@K and Recall@K 
relevant_movies = ratings.filter(col("rating") >= 4)

recommended_relevant = top_recommendations.join(
    relevant_movies,
    on="movieId"
)

precision = recommended_relevant.count() / top_recommendations.count()

recall = recommended_relevant.count() / relevant_movies.count()

print("Precision:", precision)
print("Recall:", recall)
